In [253]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [254]:
df = pd.read_csv("diabetes.csv")

In [255]:
df.shape

(768, 9)

In [256]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [257]:
X = df.drop("Outcome" , axis=1)
y = df["Outcome"]

In [258]:
df["Outcome"].value_counts(normalize=True)

Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64

In [259]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.3 , random_state=42 , stratify=y)

In [260]:
from sklearn.preprocessing import StandardScaler

std = StandardScaler()

X_train = std.fit_transform(X_train)
X_test = std.transform(X_test)

In [261]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , Sequential

In [262]:
model = Sequential([
    layers.Dense(5 , activation="relu" , input_shape=(X_train.shape[1],) ),
    layers.Dense(3 , activation="relu"),
    layers.Dense(1 , activation="sigmoid")
])
model.summary()

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67 (268.00 B)

 Trainable params: 67 (268.00 B)

 Non-trainable params: 0 (0.00 B)

In [263]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate = 0.01),
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

In [264]:
model.fit(
    X_train , y_train , 
    batch_size = 32 , 
    epochs=10 , 
    validation_split = 0.2,
    validation_data = (X_test , y_test)
)

Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6518 - loss: 0.7491 - val_accuracy: 0.6494 - val_loss: 0.6936
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6518 - loss: 0.6780 - val_accuracy: 0.6494 - val_loss: 0.6671
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6518 - loss: 0.6585 - val_accuracy: 0.6494 - val_loss: 0.6554
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6518 - loss: 0.6486 - val_accuracy: 0.6494 - val_loss: 0.6461
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6518 - loss: 0.6288 - val_accuracy: 0.6494 - val_loss: 0.5848
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6518 - loss: 0.5626 - val_accuracy: 0.6494 - val_loss: 0.5268
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6518 - loss: 0.5233 - val_accuracy: 0.6494 - val_loss: 0.5005
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6518 - loss: 0.4913 - val_accuracy: 0.6494 - val_lo

In [265]:
y_pred = model.predict(X_test)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


In [266]:
from sklearn.metrics import accuracy_score
y_pred_binary = (y_pred > 0.5).astype(int)
print(f"{accuracy_score(y_test , y_pred_binary):.3f}")

0.784


In [267]:
import keras_tuner as kt

def BuildModel(hp):

    model = Sequential([
        layers.Dense(5 , activation="relu" , input_shape = (X_train.shape[1] , )),
        layers.Dense(3 , activation="relu"),
        layers.Dense(1 , activation="sigmoid")
    ])


    model.compile(
        optimizer = hp.Choice(
            "optimizer" , values = ["adam", "sgd", "rmsprop", "adadelta"]
        ),
        loss = "binary_crossentropy",
        metrics = ["accuracy"]
    )

    return model

In [268]:
tuner = kt.RandomSearch(
    BuildModel,
    objective='val_accuracy',
    max_trials=5
)

Reloading Tuner from .\untitled_project\tuner0.json


In [269]:
tuner.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test,y_test)
)

In [270]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [271]:
model = tuner.get_best_models(num_models=1)[0]

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [272]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67 (268.00 B)

 Trainable params: 67 (268.00 B)

 Non-trainable params: 0 (0.00 B)

In [273]:
model.fit(
    X_train , y_train , 
    batch_size=32 , 
    epochs = 100 , 
    initial_epoch=6 , 
    validation_data = (X_test , y_test)
)

Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6816 - loss: 0.6469 - val_accuracy: 0.7143 - val_loss: 0.6390
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6927 - loss: 0.6244 - val_accuracy: 0.7273 - val_loss: 0.6213
Epoch 9/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7039 - loss: 0.6076 - val_accuracy: 0.7229 - val_loss: 0.6060
Epoch 10/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7058 - loss: 0.5935 - val_accuracy: 0.7229 - val_loss: 0.5926
Epoch 11/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7076 - loss: 0.5813 - val_accuracy: 0.7316 - val_loss: 0.5805
Epoch 12/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7151 - loss: 0.5707 - val_accuracy: 0.7446 - val_loss: 0.5696
Epoch 13/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7132 - loss: 0.5613 - val_accuracy: 0.7532 - val_loss: 0.5594
Epoch 14/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7151 - loss: 0.5524 - val_accuracy: 0.761